In [67]:
import torch
import torch.nn as nn


def build_mlps(c_in, mlp_channels=None, ret_before_act=False, without_norm=False):
    layers = []
    num_layers = len(mlp_channels)

    for k in range(num_layers):
        if k + 1 == num_layers and ret_before_act:
            layers.append(nn.Linear(c_in, mlp_channels[k], bias=True))
        else:
            if without_norm:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=True), nn.ReLU()]) 
            else:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=False), nn.BatchNorm1d(mlp_channels[k]), nn.ReLU()])
            c_in = mlp_channels[k]

    return nn.Sequential(*layers)



In [68]:
# Motion Transformer (MTR): https://arxiv.org/abs/2209.13508
# Published at NeurIPS 2022
# Written by Shaoshuai Shi 
# All Rights Reserved


import torch
import torch.nn as nn
import common_layers


class PointNetPolylineEncoder(nn.Module):
    def __init__(self, in_channels, hidden_dim, num_layers=3, num_pre_layers=1, out_channels=None):
        super().__init__()
        self.pre_mlps = common_layers.build_mlps(
            c_in=in_channels,
            mlp_channels=[hidden_dim] * num_pre_layers,
            ret_before_act=False
        )
        self.mlps = common_layers.build_mlps(
            c_in=hidden_dim * 2,
            mlp_channels=[hidden_dim] * (num_layers - num_pre_layers),
            ret_before_act=False
        )
        
        if out_channels is not None:
            self.out_mlps = common_layers.build_mlps(
                c_in=hidden_dim, 
                mlp_channels=[hidden_dim, out_channels], 
                ret_before_act=True, 
                without_norm=True
            )
        else:
            self.out_mlps = None 

    def forward(self, polylines, polylines_mask):
        """
        Args:
            polylines (batch_size, num_polylines, num_points_each_polylines, C):
            polylines_mask (batch_size, num_polylines, num_points_each_polylines):

        Returns:
        """
        batch_size, num_polylines,  num_points_each_polylines, C = polylines.shape
        print(polylines.shape, polylines_mask, polylines[polylines_mask].shape)
        # pre-mlp
        polylines_feature_valid = self.pre_mlps(polylines[polylines_mask])  
        print("polylines_feature_valid: ", polylines_feature_valid.shape, polylines_feature_valid)
        # (N, C) 
        # polylines[polylines_mask]：polylines 是输入的折线数据，
        # 形状为 (batch_size, num_polylines, num_points_each_polylines, C)；
        # polylines_mask 是对应的掩码，形状为 (batch_size, num_polylines, num_points_each_polylines)，
        # 它是一个布尔类型的张量，用于标记哪些点是有效的。
        polylines_feature = polylines.new_zeros(batch_size, num_polylines,  num_points_each_polylines, polylines_feature_valid.shape[-1])
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        polylines_feature[polylines_mask] = polylines_feature_valid
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        # get global feature
        a = polylines_feature.max(dim=2)
        print("a: ", a)
        pooled_feature = polylines_feature.max(dim=2)[0]
        print("pooled_feature: ", pooled_feature.shape, pooled_feature)
        polylines_feature = torch.cat((polylines_feature, pooled_feature[:, :, None, :].repeat(1, 1, num_points_each_polylines, 1)), dim=-1)
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        # mlp
        polylines_feature_valid = self.mlps(polylines_feature[polylines_mask])
        print("polylines_feature_valid: ", polylines_feature_valid.shape, polylines_feature_valid)
        feature_buffers = polylines_feature.new_zeros(batch_size, num_polylines, num_points_each_polylines, polylines_feature_valid.shape[-1])
        feature_buffers[polylines_mask] = polylines_feature_valid
        print("feature_buffers: ", feature_buffers.shape, feature_buffers)

        # max-pooling
        feature_buffers = feature_buffers.max(dim=2)[0]  # (batch_size, num_polylines, C)
        print("feature_buffers: ", feature_buffers.shape, feature_buffers)
        # out-mlp 
        if self.out_mlps is not None:
            valid_mask = (polylines_mask.sum(dim=-1) > 0)
            feature_buffers_valid = self.out_mlps(feature_buffers[valid_mask])  # (N, C)
            feature_buffers = feature_buffers.new_zeros(batch_size, num_polylines, feature_buffers_valid.shape[-1])
            feature_buffers[valid_mask] = feature_buffers_valid
        return feature_buffers


In [69]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 定义测试数据集类
class TestDataset(Dataset):
    def __init__(self, num_samples, num_polylines, num_points_each_polylines, in_channels):
        self.num_samples = num_samples
        self.num_polylines = num_polylines
        self.num_points_each_polylines = num_points_each_polylines
        self.in_channels = in_channels

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        polylines = torch.randn(self.num_polylines, self.num_points_each_polylines, self.in_channels)
        polylines_mask = torch.randint(0, 2, (self.num_polylines, self.num_points_each_polylines)).bool()
        return polylines, polylines_mask


# 定义测试函数
def test_PointNetPolylineEncoder():
    in_channels = 5
    hidden_dim = 10
    num_layers = 3
    num_pre_layers = 1
    out_channels = 8
    batch_size = 1

    # 创建 PointNetPolylineEncoder 实例
    encoder = PointNetPolylineEncoder(
        in_channels=in_channels,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_pre_layers=num_pre_layers,
        out_channels=out_channels
    )

    # 创建测试数据集和数据加载器
    dataset = TestDataset(
        num_samples=batch_size,
        num_polylines=3,
        num_points_each_polylines=4,
        in_channels=in_channels
    )
    dataloader = DataLoader(dataset, batch_size=batch_size)

    # 进行测试
    for polylines, polylines_mask in dataloader:
        print("Input shape:", polylines.shape, polylines_mask.shape)
        output = encoder(polylines, polylines_mask)
        print("Output shape:", output.shape)
        assert len(output.shape) == 3
        assert output.shape[0] == batch_size
        break


if __name__ == "__main__":
    test_PointNetPolylineEncoder()

Input shape: torch.Size([1, 3, 4, 5]) torch.Size([1, 3, 4])
torch.Size([1, 3, 4, 5]) tensor([[[ True,  True,  True,  True],
         [False,  True,  True, False],
         [False, False,  True,  True]]]) torch.Size([8, 5])
polylines_feature_valid:  torch.Size([8, 10]) tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.6106, 0.0000, 0.5504, 0.0000, 0.0000,
         0.2465],
        [0.0038, 0.7503, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 1.1268, 0.0000, 0.0000, 0.7900, 0.5135, 0.0000, 0.7653, 0.0000,
         0.4380],
        [1.7365, 0.2015, 0.7275, 0.2473, 0.0000, 0.0000, 0.0000, 0.0000, 0.4891,
         0.0000],
        [0.4417, 0.0823, 0.0000, 0.2954, 1.9453, 0.1493, 0.0000, 2.3146, 1.9662,
         2.2552],
        [0.0000, 0.4287, 0.1125, 0.0789, 0.0000, 1.3151, 1.7056, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.5683, 0.0000, 0.0000, 0.1529, 1.2012, 1.1460, 0.0000, 0.0000,
         0.0000],
        [1.1399, 0.0000, 2.1617, 2.285

In [70]:
import torch
import torch.nn as nn


def build_mlps(c_in, mlp_channels=None, ret_before_act=False, without_norm=False):
    layers = []
    num_layers = len(mlp_channels)

    for k in range(num_layers):
        if k + 1 == num_layers and ret_before_act:
            layers.append(nn.Linear(c_in, mlp_channels[k], bias=True))
        else:
            if without_norm:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=True), nn.ReLU()]) 
            else:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=False), nn.BatchNorm1d(mlp_channels[k]), nn.ReLU()])
            c_in = mlp_channels[k]

    return nn.Sequential(*layers)



In [71]:
import unittest
import torch
import torch.nn as nn


class MyClass:
    def __init__(self, num_future_frames):
        self.num_future_frames = num_future_frames

    def build_dense_future_prediction_layers(self, hidden_dim, num_future_frames):
        self.obj_pos_encoding_layer = build_mlps(
            c_in=2, mlp_channels=[hidden_dim, hidden_dim, hidden_dim], ret_before_act=True, without_norm=True
        )
        self.dense_future_head = build_mlps(
            c_in=hidden_dim * 2,
            mlp_channels=[hidden_dim, hidden_dim, num_future_frames * 7], ret_before_act=True
        )

        self.future_traj_mlps = build_mlps(
            c_in=4 * self.num_future_frames, mlp_channels=[hidden_dim, hidden_dim, hidden_dim], ret_before_act=True,
            without_norm=True
        )
        self.traj_fusion_mlps = build_mlps(
            c_in=hidden_dim * 2, mlp_channels=[hidden_dim, hidden_dim, hidden_dim], ret_before_act=True,
            without_norm=True
        )

def build_mlps(c_in, mlp_channels=None, ret_before_act=False, without_norm=False):
    layers = []
    num_layers = len(mlp_channels)

    for k in range(num_layers):
        if k + 1 == num_layers and ret_before_act:
            layers.append(nn.Linear(c_in, mlp_channels[k], bias=True))
        else:
            if without_norm:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=True), nn.ReLU()])
            else:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=False), nn.BatchNorm1d(mlp_channels[k]), nn.ReLU()])
            c_in = mlp_channels[k]

    return nn.Sequential(*layers)


class TestBuildDenseFuturePredictionLayers(unittest.TestCase):
    def test_build_dense_future_prediction_layers(self):
        num_future_frames = 5
        hidden_dim = 64
        test_obj = MyClass(num_future_frames)
        test_obj.build_dense_future_prediction_layers(hidden_dim, num_future_frames)

        self.assertEqual(isinstance(test_obj.obj_pos_encoding_layer, nn.Sequential), True)
        self.assertEqual(isinstance(test_obj.dense_future_head, nn.Sequential), True)
        self.assertEqual(isinstance(test_obj.future_traj_mlps, nn.Sequential), True)
        self.assertEqual(isinstance(test_obj.traj_fusion_mlps, nn.Sequential), True)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestBuildDenseFuturePredictionLayers)
    runner = unittest.TextTestRunner()
    runner.run(suite)

.
----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


In [72]:
import torch

# 假设的参数
num_center_objects = 3
obj_polylines_feature = torch.tensor([
    [[1, 2], [3, 4], [5, 6]],
    [[7, 8], [9, 10], [11, 12]],
    [[13, 14], [15, 16], [17, 18]]
])
track_index_to_predict = torch.tensor([0, 1, 2])

# 执行代码
center_objects_feature = obj_polylines_feature[torch.arange(num_center_objects), track_index_to_predict]

print("选取的中心物体特征:")
print(center_objects_feature)

选取的中心物体特征:
tensor([[ 1,  2],
        [ 9, 10],
        [17, 18]])


In [73]:
import unittest
import torch
import torch.nn as nn
from unittest.mock import patch


class MyClass:
    def __init__(self):
        self.num_future_frames = 5
        self.obj_pos_encoding_layer = nn.Linear(2, 10)
        self.dense_future_head = nn.Linear(20, self.num_future_frames * 7)
        self.future_traj_mlps = nn.Linear(4 * self.num_future_frames, 10)
        self.traj_fusion_mlps = nn.Linear(20, 10)
        self.forward_ret_dict = {}

    def apply_dense_future_prediction(self, obj_feature, obj_mask, obj_pos):
        num_center_objects, num_objects, _ = obj_feature.shape
        print(obj_pos[obj_mask])
        # dense future prediction
        obj_pos_valid = obj_pos[obj_mask][..., 0:2]
        print("obj_pos_valid shape:", obj_pos_valid.shape)
        print("obj_pos_valid value:", obj_pos_valid)
        obj_feature_valid = obj_feature[obj_mask]
        print("obj_feature_valid shape:", obj_feature_valid.shape)
        print("obj_feature_valid value:", obj_feature_valid)
        obj_pos_feature_valid = self.obj_pos_encoding_layer(obj_pos_valid)
        print("obj_pos_feature_valid shape:", obj_pos_feature_valid.shape)
        print("obj_pos_feature_valid value:", obj_pos_feature_valid)
        obj_fused_feature_valid = torch.cat((obj_pos_feature_valid, obj_feature_valid), dim=-1)
        print("obj_fused_feature_valid shape:", obj_fused_feature_valid.shape)
        print("obj_fused_feature_valid value:", obj_fused_feature_valid)

        pred_dense_trajs_valid = self.dense_future_head(obj_fused_feature_valid)
        print("pred_dense_trajs_valid shape before view:", pred_dense_trajs_valid.shape)
        print("pred_dense_trajs_valid value before view:", pred_dense_trajs_valid)
        pred_dense_trajs_valid = pred_dense_trajs_valid.view(pred_dense_trajs_valid.shape[0], self.num_future_frames, -1)
        print("pred_dense_trajs_valid shape after view:", pred_dense_trajs_valid.shape)
        print("pred_dense_trajs_valid value after view:", pred_dense_trajs_valid)

        temp_center = pred_dense_trajs_valid[:, :, 0:2] + obj_pos_valid[:, None, 0:2]
        print("temp_center shape:", temp_center.shape)
        print("temp_center value:", temp_center)
        pred_dense_trajs_valid = torch.cat((temp_center, pred_dense_trajs_valid[:, :, 2:]), dim=-1)
        print("pred_dense_trajs_valid shape after cat:", pred_dense_trajs_valid.shape)
        print("pred_dense_trajs_valid value after cat:", pred_dense_trajs_valid)

        # future feature encoding and fuse to past obj_feature
        obj_future_input_valid = pred_dense_trajs_valid[:, :, [0, 1, -2, -1]].flatten(start_dim=1, end_dim=2)  
        print("obj_future_input_valid shape:", obj_future_input_valid.shape)
        print("obj_future_input_valid value:", obj_future_input_valid)
        # pred_dense_trajs_valid 是一个张量，它应该是之前通过模型计算得到的密集未来轨迹预测的有效部分。
        # [:, :, [0, 1, -2, -1]] 这部分是对 pred_dense_trajs_valid 进行索引操作。
        # [:, :] 表示选取所有的样本（第一维）和所有的时间步（第二维）。
        # [0, 1, -2, -1] 表示在第三维（特征维度）上选取索引为 0、1、倒数第 2 和倒数第 1 的特征。这样做可能是为了提取特定的特征分量，例如可能与位置和方向等相关的特征。
        # .flatten(start_dim=1, end_dim=2) 是对选取后的张量进行展平操作。
        # start_dim=1 和 end_dim=2 表示从第 1 维（索引从 0 开始计数）到第 2 维之间的维度进行展平。也就是说，它会将时间步和选取的特征维度进行合并，从而改变张量的形状。
        obj_future_feature_valid = self.future_traj_mlps(obj_future_input_valid)
        print("obj_future_feature_valid shape:", obj_future_feature_valid.shape)
        print("obj_future_feature_valid value:", obj_future_feature_valid)

        obj_full_trajs_feature = torch.cat((obj_feature_valid, obj_future_feature_valid), dim=-1)
        print("obj_full_trajs_feature shape:", obj_full_trajs_feature.shape)
        print("obj_full_trajs_feature value:", obj_full_trajs_feature)
        obj_feature_valid = self.traj_fusion_mlps(obj_full_trajs_feature)
        print("obj_feature_valid shape:", obj_feature_valid.shape)
        print("obj_feature_valid value:", obj_feature_valid)

        ret_obj_feature = torch.zeros_like(obj_feature)
        print("ret_obj_feature shape before assignment:", ret_obj_feature.shape)
        print("ret_obj_feature value before assignment:", ret_obj_feature)
        ret_obj_feature[obj_mask] = obj_feature_valid
        print("ret_obj_feature shape after assignment:", ret_obj_feature.shape)
        print("ret_obj_feature value after assignment:", ret_obj_feature)

        ret_pred_dense_future_trajs = obj_feature.new_zeros(num_center_objects, num_objects, self.num_future_frames, 7)
        print("ret_pred_dense_future_trajs shape before assignment:", ret_pred_dense_future_trajs.shape)
        print("ret_pred_dense_future_trajs value before assignment:", ret_pred_dense_future_trajs)
        ret_pred_dense_future_trajs[obj_mask] = pred_dense_trajs_valid
        print("ret_pred_dense_future_trajs shape after assignment:", ret_pred_dense_future_trajs.shape)
        print("ret_pred_dense_future_trajs value after assignment:", ret_pred_dense_future_trajs)
        self.forward_ret_dict['pred_dense_trajs'] = ret_pred_dense_future_trajs

        return ret_obj_feature, ret_pred_dense_future_trajs


class TestApplyDenseFuturePrediction(unittest.TestCase):
    def test_apply_dense_future_prediction(self):
        # 创建测试数据
        num_center_objects = 2
        num_objects = 3
        feature_dim = 10
        obj_feature = torch.randn(num_center_objects, num_objects, feature_dim)
        obj_mask = torch.tensor([[True, False, True], [False, True, False]])
        obj_pos = torch.randn(num_center_objects, num_objects, 3)
        print(obj_feature.shape, obj_mask, obj_pos.shape)

        # 创建类的实例
        instance = MyClass()

        # 调用函数
        ret_obj_feature, ret_pred_dense_future_trajs = instance.apply_dense_future_prediction(obj_feature, obj_mask, obj_pos)

        # 检查返回值的类型
        self.assertEqual(isinstance(ret_obj_feature, torch.Tensor), True)
        self.assertEqual(isinstance(ret_pred_dense_future_trajs, torch.Tensor), True)

        # 检查返回值的形状
        self.assertEqual(ret_obj_feature.shape, obj_feature.shape)
        self.assertEqual(ret_pred_dense_future_trajs.shape, (num_center_objects, num_objects, instance.num_future_frames, 7))

        # 检查 forward_ret_dict 是否更新
        self.assertEqual('pred_dense_trajs' in instance.forward_ret_dict, True)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestApplyDenseFuturePrediction)
    runner = unittest.TextTestRunner()
    runner.run(suite)

.

torch.Size([2, 3, 10]) tensor([[ True, False,  True],
        [False,  True, False]]) torch.Size([2, 3, 3])
tensor([[-1.2578, -0.0969, -0.3278],
        [ 3.0822, -0.9925,  1.3769],
        [-0.4399,  0.0223, -0.8351]])
obj_pos_valid shape: torch.Size([3, 2])
obj_pos_valid value: tensor([[-1.2578, -0.0969],
        [ 3.0822, -0.9925],
        [-0.4399,  0.0223]])
obj_feature_valid shape: torch.Size([3, 10])
obj_feature_valid value: tensor([[-0.0433,  0.8961, -0.9104,  0.3930, -1.0999, -0.9285,  0.5192, -0.3109,
          1.1839,  0.6098],
        [-0.1529, -0.0214,  1.2477,  1.4585,  0.1734,  0.7293, -1.0122, -0.8814,
          1.1603,  1.3575],
        [-1.0869,  0.8074,  1.7521,  0.8761, -0.0270, -1.7078, -0.6210,  0.0423,
          1.1989, -0.5275]])
obj_pos_feature_valid shape: torch.Size([3, 10])
obj_pos_feature_valid value: tensor([[-0.2663, -0.9568,  0.0611, -0.1231, -1.0448,  0.3417, -0.8001,  0.6381,
         -0.3689,  1.0023],
        [ 1.3884,  0.5442,  1.6258,  0.2822,  1.4


----------------------------------------------------------------------
Ran 1 test in 0.006s

OK


In [74]:
import unittest
import torch
from unittest.mock import patch


# 模拟 position_encoding_utils 模块
class position_encoding_utils:
    @staticmethod
    def gen_sineembed_for_position(pos, hidden_dim):
        # 简单返回一个与输入形状相关的随机张量作为模拟结果
        return torch.randn(pos.size(0), pos.size(1), hidden_dim)


class MyClass:
    def __init__(self):
        self.use_place_holder = False
        self.d_model = 64
        # 模拟 intention_points
        self.intention_points = {
            'type1': torch.randn(10, 2),
            'type2': torch.randn(10, 2)
        }
        # 模拟 intention_query_mlps
        self.intention_query_mlps = torch.nn.Linear(self.d_model, self.d_model)

    def get_motion_query(self, center_objects_type):
        num_center_objects = len(center_objects_type)
        print("num_center_objects:", num_center_objects)
        if self.use_place_holder:
            raise NotImplementedError
        else:
            intention_points = torch.stack([
                self.intention_points[center_objects_type[obj_idx]]
                for obj_idx in range(num_center_objects)], dim=0)
            print("intention_points after stack shape:", intention_points.shape)
            print("intention_points after stack value:", intention_points)
            intention_points = intention_points.permute(1, 0, 2)  # (num_query, num_center_objects, 2)
            print("intention_points after permute shape:", intention_points.shape)
            print("intention_points after permute value:", intention_points)

            intention_query = position_encoding_utils.gen_sineembed_for_position(intention_points, hidden_dim=self.d_model)
            print("intention_query after gen_sineembed shape:", intention_query.shape)
            print("intention_query after gen_sineembed value:", intention_query)
            intention_query = self.intention_query_mlps(intention_query.view(-1, self.d_model)).view(-1, num_center_objects, self.d_model)  # (num_query, num_center_objects, C)
            print("intention_query after mlps shape:", intention_query.shape)
            print("intention_query after mlps value:", intention_query)
        return intention_query, intention_points


class TestGetMotionQuery(unittest.TestCase):
    def test_get_motion_query(self):
        # 创建测试数据
        center_objects_type = ['type1', 'type2']
        # 创建类的实例
        instance = MyClass()
        # 调用函数
        intention_query, intention_points = instance.get_motion_query(center_objects_type)
        # 检查返回值的类型
        self.assertEqual(isinstance(intention_query, torch.Tensor), True)
        self.assertEqual(isinstance(intention_points, torch.Tensor), True)
        # 检查返回值的形状
        num_query = instance.intention_points[center_objects_type[0]].size(0)
        num_center_objects = len(center_objects_type)
        self.assertEqual(intention_query.shape, (num_query, num_center_objects, instance.d_model))
        self.assertEqual(intention_points.shape, (num_query, num_center_objects, 2))


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestGetMotionQuery)
    runner = unittest.TextTestRunner()
    runner.run(suite)

.

num_center_objects: 2
intention_points after stack shape: torch.Size([2, 10, 2])
intention_points after stack value: tensor([[[-1.2519,  0.6938],
         [ 0.4454,  0.7613],
         [ 1.0201, -0.0801],
         [-0.1952,  1.2070],
         [ 1.3538,  1.1293],
         [ 1.3163,  0.5850],
         [-1.3998,  0.0090],
         [ 1.0969,  1.9102],
         [-0.9391,  1.3864],
         [ 1.6610, -1.3226]],

        [[-0.7691,  0.7158],
         [ 1.1709, -0.0365],
         [-2.0086,  1.0109],
         [-1.2543, -1.2272],
         [-0.4816,  0.3639],
         [ 0.7092, -1.0749],
         [ 0.0699, -0.1238],
         [-0.3600,  2.9879],
         [ 0.4610,  1.5291],
         [-0.1583, -1.1332]]])
intention_points after permute shape: torch.Size([10, 2, 2])
intention_points after permute value: tensor([[[-1.2519,  0.6938],
         [-0.7691,  0.7158]],

        [[ 0.4454,  0.7613],
         [ 1.1709, -0.0365]],

        [[ 1.0201, -0.0801],
         [-2.0086,  1.0109]],

        [[-0.1952,  


----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


In [75]:
import torch
import torch.nn as nn

# 模拟 position_encoding_utils.gen_sineembed_for_position 函数
class position_encoding_utils:
    @staticmethod
    def gen_sineembed_for_position(pos, hidden_dim):
        # 简单返回一个全零张量作为示例
        return torch.zeros(pos.shape[:-1] + (hidden_dim,), device=pos.device)

# 模拟 attention_layer 类
class AttentionLayer(nn.Module):
    def __init__(self, d_model):
        super(AttentionLayer, self).__init__()
        self.linear = nn.Linear(d_model, d_model)

    def forward(self, tgt, query_pos, query_sine_embed, memory, memory_key_padding_mask=None, pos=None, is_first=False,
                memory_valid_mask=None, key_batch_cnt=None, index_pair=None, index_pair_batch=None):
        # 简单返回 tgt 经过线性层的结果作为示例
        return self.linear(tgt)

class TestClass:
    def apply_cross_attention(self, kv_feature, kv_mask, kv_pos, query_content, query_embed, attention_layer,
                              dynamic_query_center=None, layer_idx=0, use_local_attn=False, query_index_pair=None,
                              query_content_pre_mlp=None, query_embed_pre_mlp=None):
        """
        Args:
            kv_feature (B, N, C):
            kv_mask (B, N):
            kv_pos (B, N, 3):
            query_content (M, B, C):
            query_embed (M, B, C):
            dynamic_query_center (M, B, 2): . Defaults to None.
            attention_layer (layer):

            query_index_pair (B, M, K)

        Returns:
            attended_features: (B, M, C)
            attn_weights:
        """
        if query_content_pre_mlp is not None:
            print("Before query_content_pre_mlp, shape:", query_content.shape)
            print("Before query_content_pre_mlp, value:", query_content)
            query_content = query_content_pre_mlp(query_content)
            print("After query_content_pre_mlp, shape:", query_content.shape)
            print("After query_content_pre_mlp, value:", query_content)
        if query_embed_pre_mlp is not None:
            print("Before query_embed_pre_mlp, shape:", query_embed.shape)
            print("Before query_embed_pre_mlp, value:", query_embed)
            query_embed = query_embed_pre_mlp(query_embed)
            print("After query_embed_pre_mlp, shape:", query_embed.shape)
            print("After query_embed_pre_mlp, value:", query_embed)

        num_q, batch_size, d_model = query_content.shape
        print("query_content shape:", query_content.shape)
        print("query_content value:", query_content)
        print("dynamic_query_center shape:", dynamic_query_center.shape)
        print("dynamic_query_center value:", dynamic_query_center)
        searching_query = position_encoding_utils.gen_sineembed_for_position(dynamic_query_center, hidden_dim=d_model)
        print("searching_query shape:", searching_query.shape)
        print("searching_query value:", searching_query)
        print("kv_pos shape:", kv_pos.shape)
        print("kv_pos value:", kv_pos)
        kv_pos = kv_pos.permute(1, 0, 2)[:, :, 0:2]
        print("permuted kv_pos shape:", kv_pos.shape)
        print("permuted kv_pos value:", kv_pos)
        kv_pos_embed = position_encoding_utils.gen_sineembed_for_position(kv_pos, hidden_dim=d_model)
        print("kv_pos_embed shape:", kv_pos_embed.shape)
        print("kv_pos_embed value:", kv_pos_embed)

        if not use_local_attn:
            print("Not using local attention")
            print("kv_feature shape:", kv_feature.shape)
            print("kv_feature value:", kv_feature)
            print("kv_mask shape:", kv_mask.shape)
            print("kv_mask value:", kv_mask)
            query_feature = attention_layer(
                tgt=query_content,
                query_pos=query_embed,
                query_sine_embed=searching_query,
                memory=kv_feature.permute(1, 0, 2),
                memory_key_padding_mask=~kv_mask,
                pos=kv_pos_embed,
                is_first=(layer_idx == 0)
            )  # (M, B, C)
            print("query_feature shape:", query_feature.shape)
            print("query_feature value:", query_feature)
        else:
            print("Using local attention")
            batch_size, num_kv, _ = kv_feature.shape
            print("kv_feature shape:", kv_feature.shape)
            print("kv_feature value:", kv_feature)
            kv_feature_stack = kv_feature.flatten(start_dim=0, end_dim=1)
            print("kv_feature_stack shape:", kv_feature_stack.shape)
            print("kv_feature_stack value:", kv_feature_stack)
            print("kv_pos_embed shape:", kv_pos_embed.shape)
            print("kv_pos_embed value:", kv_pos_embed)
            kv_pos_embed_stack = kv_pos_embed.permute(1, 0, 2).contiguous().flatten(start_dim=0, end_dim=1)
            print("kv_pos_embed_stack shape:", kv_pos_embed_stack.shape)
            print("kv_pos_embed_stack value:", kv_pos_embed_stack)
            print("kv_mask shape:", kv_mask.shape)
            print("kv_mask value:", kv_mask)
            kv_mask_stack = kv_mask.view(-1)
            print("kv_mask_stack shape:", kv_mask_stack.shape)
            print("kv_mask_stack value:", kv_mask_stack)

            key_batch_cnt = num_kv * torch.ones(batch_size).int().to(kv_feature.device)
            print("key_batch_cnt shape:", key_batch_cnt.shape)
            print("key_batch_cnt value:", key_batch_cnt)
            print("query_index_pair shape:", query_index_pair.shape)
            print("query_index_pair value:", query_index_pair)
            query_index_pair = query_index_pair.view(batch_size * num_q, -1)
            print("reshaped query_index_pair shape:", query_index_pair.shape)
            print("reshaped query_index_pair value:", query_index_pair)
            index_pair_batch = torch.arange(batch_size).type_as(key_batch_cnt)[:, None].repeat(1, num_q).view(-1)  # (batch_size * num_q)
            print("index_pair_batch shape:", index_pair_batch.shape)
            print("index_pair_batch value:", index_pair_batch)
            assert len(query_index_pair) == len(index_pair_batch)

            query_feature = attention_layer(
                tgt=query_content,
                query_pos=query_embed,
                query_sine_embed=searching_query,
                memory=kv_feature_stack,
                memory_valid_mask=kv_mask_stack,
                pos=kv_pos_embed_stack,
                is_first=(layer_idx == 0),
                key_batch_cnt=key_batch_cnt,
                index_pair=query_index_pair,
                index_pair_batch=index_pair_batch
            )
            print("query_feature shape before reshape:", query_feature.shape)
            print("query_feature value before reshape:", query_feature)
            query_feature = query_feature.view(batch_size, num_q, d_model).permute(1, 0, 2)  # (M, B, C)
            print("query_feature shape after reshape:", query_feature.shape)
            print("query_feature value after reshape:", query_feature)

        return query_feature

# 测试代码
if __name__ == "__main__":
    # 定义参数
    B = 2  # 批次大小
    N = 3  # kv 特征数量
    M = 4  # 查询数量
    C = 5  # 特征维度
    K = 2  # 局部注意力的 K 值

    # 生成输入数据
    kv_feature = torch.randn(B, N, C)
    kv_mask = torch.randn(B, N) > 0
    kv_pos = torch.randn(B, N, 3)
    query_content = torch.randn(M, B, C)
    query_embed = torch.randn(M, B, C)
    dynamic_query_center = torch.randn(M, B, 2)
    attention_layer = AttentionLayer(C)
    query_index_pair = torch.randint(0, N, (B, M, K))

    # 创建测试类实例
    test_obj = TestClass()

    # 测试不使用局部注意力
    output_without_local_attn = test_obj.apply_cross_attention(
        kv_feature, kv_mask, kv_pos, query_content, query_embed, attention_layer,
        dynamic_query_center=dynamic_query_center, layer_idx=0, use_local_attn=False, query_index_pair=query_index_pair
    )
    print("Output without local attention shape:", output_without_local_attn.shape)

    # 测试使用局部注意力
    output_with_local_attn = test_obj.apply_cross_attention(
        kv_feature, kv_mask, kv_pos, query_content, query_embed, attention_layer,
        dynamic_query_center=dynamic_query_center, layer_idx=0, use_local_attn=True, query_index_pair=query_index_pair
    )
    print("Output with local attention shape:", output_with_local_attn.shape)


query_content shape: torch.Size([4, 2, 5])
query_content value: tensor([[[-0.1540,  0.5471, -1.3060, -0.2755,  1.6831],
         [ 0.0379,  0.7951, -0.3258,  0.6014, -1.2816]],

        [[ 0.0421,  0.3754, -0.7446,  0.3918,  1.3842],
         [-0.1624,  0.0580,  1.4864,  0.9803, -0.7413]],

        [[ 0.5467,  1.3699,  0.4209,  1.9268, -0.8666],
         [-0.1619,  1.4319, -0.5112,  0.7889, -1.8368]],

        [[ 1.6285, -0.6402,  1.0281,  0.5259,  0.1289],
         [-0.6222, -0.5774, -1.2777, -0.3950,  0.4234]]])
dynamic_query_center shape: torch.Size([4, 2, 2])
dynamic_query_center value: tensor([[[-0.5406, -1.2174],
         [-0.3121, -1.3312]],

        [[ 0.0278, -1.5814],
         [ 0.4509,  1.1568]],

        [[ 0.6962, -0.6782],
         [ 1.4878,  0.1283]],

        [[-1.0791,  0.6519],
         [ 0.2885,  0.1829]]])
searching_query shape: torch.Size([4, 2, 5])
searching_query value: tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],

In [76]:
import torch

# 模拟基础和动态多段线索引
base_map_idxs = torch.tensor([[[1, 2, 3], [4, 5, 6]]])
dynamic_map_idxs = torch.tensor([[[3, 7, 8], [9, 10, 11]]])

# 合并基础和动态多段线索引
collected_idxs = torch.cat((base_map_idxs, dynamic_map_idxs), dim=-1)
print("合并后的索引 collected_idxs 形状:", collected_idxs.shape)
print("合并后的索引 collected_idxs 值:", collected_idxs)

# 去重合并后的多段线索引
sorted_idxs = collected_idxs.sort(dim=-1)[0]
print("111", collected_idxs.sort(dim=-1))
print("排序后的索引 sorted_idxs 形状:", sorted_idxs.shape)
print("排序后的索引 sorted_idxs 值:", sorted_idxs)

duplicate_mask_slice = (sorted_idxs[..., 1:] - sorted_idxs[..., :-1] != 0)  # 检测重复索引
print("检测重复索引的掩码切片 duplicate_mask_slice 形状:", duplicate_mask_slice.shape)
print("检测重复索引的掩码切片 duplicate_mask_slice 值:", duplicate_mask_slice)

duplicate_mask = torch.ones_like(collected_idxs).bool()
duplicate_mask[..., 1:] = duplicate_mask_slice
print("完整的重复索引掩码 duplicate_mask 形状:", duplicate_mask.shape)
print("完整的重复索引掩码 duplicate_mask 值:", duplicate_mask)

sorted_idxs[~duplicate_mask] = -1  # 将重复索引标记为无效
print("去重并标记后的索引 sorted_idxs 形状:", sorted_idxs.shape)
print("去重并标记后的索引 sorted_idxs 值:", sorted_idxs)

合并后的索引 collected_idxs 形状: torch.Size([1, 2, 6])
合并后的索引 collected_idxs 值: tensor([[[ 1,  2,  3,  3,  7,  8],
         [ 4,  5,  6,  9, 10, 11]]])
111 torch.return_types.sort(
values=tensor([[[ 1,  2,  3,  3,  7,  8],
         [ 4,  5,  6,  9, 10, 11]]]),
indices=tensor([[[0, 1, 2, 3, 4, 5],
         [0, 1, 2, 3, 4, 5]]]))
排序后的索引 sorted_idxs 形状: torch.Size([1, 2, 6])
排序后的索引 sorted_idxs 值: tensor([[[ 1,  2,  3,  3,  7,  8],
         [ 4,  5,  6,  9, 10, 11]]])
检测重复索引的掩码切片 duplicate_mask_slice 形状: torch.Size([1, 2, 5])
检测重复索引的掩码切片 duplicate_mask_slice 值: tensor([[[ True,  True, False,  True,  True],
         [ True,  True,  True,  True,  True]]])
完整的重复索引掩码 duplicate_mask 形状: torch.Size([1, 2, 6])
完整的重复索引掩码 duplicate_mask 值: tensor([[[ True,  True,  True, False,  True,  True],
         [ True,  True,  True,  True,  True,  True]]])
去重并标记后的索引 sorted_idxs 形状: torch.Size([1, 2, 6])
去重并标记后的索引 sorted_idxs 值: tensor([[[ 1,  2,  3, -1,  7,  8],
         [ 4,  5,  6,  9, 10, 11]]])


In [77]:
import torch
import torch.nn as nn

class TestModel(nn.Module):
    def __init__(self, layer_idx, num_future_frames):
        super(TestModel, self).__init__()
        self.layer_idx = layer_idx
        self.num_future_frames = num_future_frames
        # 模拟 motion_cls_heads
        self.motion_cls_heads = nn.ModuleList([nn.Linear(10, 1) for _ in range(5)])
        # 模拟 motion_reg_heads
        self.motion_reg_heads = nn.ModuleList([nn.Linear(10, num_future_frames * 5) for _ in range(5)])
        # 模拟 motion_vel_heads
        self.motion_vel_heads = nn.ModuleList([nn.Linear(10, num_future_frames * 2) for _ in range(5)])

    def forward(self, query_content, num_center_objects, num_query):
        pred_list = []
        print("输入的 query_content 形状:", query_content.shape)
        print("输入的 query_content 值:", query_content)

        # motion prediction
        query_content_t = query_content.permute(1, 0, 2).contiguous().view(num_center_objects * num_query, -1)
        print("经过处理后的 query_content_t 形状:", query_content_t.shape)
        print("经过处理后的 query_content_t 值:", query_content_t)

        pred_scores = self.motion_cls_heads[self.layer_idx](query_content_t).view(num_center_objects, num_query)
        print("预测分数 pred_scores 形状:", pred_scores.shape)
        print("预测分数 pred_scores 值:", pred_scores)

        if self.motion_vel_heads is not None:
            pred_trajs = self.motion_reg_heads[self.layer_idx](query_content_t).view(num_center_objects, num_query, self.num_future_frames, 5)
            print("预测轨迹 pred_trajs 初始形状:", pred_trajs.shape)
            print("预测轨迹 pred_trajs 初始值:", pred_trajs)

            pred_vel = self.motion_vel_heads[self.layer_idx](query_content_t).view(num_center_objects, num_query, self.num_future_frames, 2)
            print("预测速度 pred_vel 形状:", pred_vel.shape)
            print("预测速度 pred_vel 值:", pred_vel)

            pred_trajs = torch.cat((pred_trajs, pred_vel), dim=-1)
            print("拼接后的预测轨迹 pred_trajs 形状:", pred_trajs.shape)
            print("拼接后的预测轨迹 pred_trajs 值:", pred_trajs)
        else:
            pred_trajs = self.motion_reg_heads[self.layer_idx](query_content_t).view(num_center_objects, num_query, self.num_future_frames, 7)
            print("预测轨迹 pred_trajs 形状:", pred_trajs.shape)
            print("预测轨迹 pred_trajs 值:", pred_trajs)

        pred_list.append([pred_scores, pred_trajs])

        # update
        pred_waypoints = pred_trajs[:, :, :, 0:2]
        print("预测路径点 pred_waypoints 形状:", pred_waypoints.shape)
        print("预测路径点 pred_waypoints 值:", pred_waypoints)

        dynamic_query_center = pred_trajs[:, :, -1, 0:2].contiguous().permute(1, 0, 2)  # (num_query, num_center_objects, 2)
        print("动态查询中心 dynamic_query_center 形状:", dynamic_query_center.shape)
        print("动态查询中心 dynamic_query_center 值:", dynamic_query_center)

        return pred_list, pred_waypoints, dynamic_query_center

# 模拟输入数据
num_center_objects = 2
num_query = 3
num_future_frames = 4
query_content = torch.randn(num_query, num_center_objects, 10)
layer_idx = 0

# 创建模型实例
model = TestModel(layer_idx, num_future_frames)

# 运行代码
pred_list, pred_waypoints, dynamic_query_center = model(query_content, num_center_objects, num_query)

输入的 query_content 形状: torch.Size([3, 2, 10])
输入的 query_content 值: tensor([[[-2.7558,  0.1217,  0.8761, -1.1406,  1.2531, -0.9577, -1.1015,
          -0.1475,  0.3692, -1.4324],
         [ 0.4002,  2.4671, -0.4698,  0.0897,  0.6092,  1.3034, -0.7067,
          -0.4403, -0.1839, -0.6094]],

        [[ 1.4620,  0.1986,  0.4721,  0.9273, -0.1187,  0.6980,  0.1432,
          -1.0034,  0.8317, -0.1859],
         [ 0.5862,  1.3914, -1.0076, -0.7251,  0.9169,  0.5216,  0.5050,
          -0.1631,  0.7902, -0.8142]],

        [[-1.0568,  0.8589,  0.8892, -0.3001,  0.0955, -0.7554, -0.3871,
           1.1794,  1.9120, -0.9109],
         [ 0.7715, -0.9064,  0.6280,  0.2278,  0.8261, -0.4259,  0.5353,
           0.1545,  1.0818,  0.0391]]])
经过处理后的 query_content_t 形状: torch.Size([6, 10])
经过处理后的 query_content_t 值: tensor([[-2.7558,  0.1217,  0.8761, -1.1406,  1.2531, -0.9577, -1.1015, -0.1475,
          0.3692, -1.4324],
        [ 1.4620,  0.1986,  0.4721,  0.9273, -0.1187,  0.6980,  0.1432, -1.0034,

In [103]:
import torch


# 模拟一个类，假设 get_decoder_loss 方法在这个类中
class MockClass:
    def __init__(self):
        self.forward_ret_dict = {}

    def get_decoder_loss(self):
        center_gt_trajs = self.forward_ret_dict['center_gt_trajs'].cuda()
        center_gt_trajs_mask = self.forward_ret_dict['center_gt_trajs_mask'].cuda()
        center_gt_final_valid_idx = self.forward_ret_dict['center_gt_final_valid_idx'].long()
        assert center_gt_trajs.shape[-1] == 4
        return None, None, None


# 测试正常情况
def test_normal_case():
    mock_obj = MockClass()

    num_center_objects = 3  # N
    max_num_gt = 4  # G
    num_future_timestamps = 5  # T

    # 模拟数据
    center_gt_trajs = torch.randn(num_center_objects, max_num_gt, num_future_timestamps, 4)
    center_gt_trajs_mask = torch.randint(0, 2, (num_center_objects, max_num_gt, num_future_timestamps)).bool()
    center_gt_final_valid_idx = torch.randint(0, max_num_gt, (num_center_objects,))
    print(center_gt_trajs, center_gt_trajs_mask, center_gt_final_valid_idx)

    mock_obj.forward_ret_dict['center_gt_trajs'] = center_gt_trajs
    mock_obj.forward_ret_dict['center_gt_trajs_mask'] = center_gt_trajs_mask
    mock_obj.forward_ret_dict['center_gt_final_valid_idx'] = center_gt_final_valid_idx

    try:
        mock_obj.get_decoder_loss()
        print("正常情况测试通过：数据形状符合预期，断言未触发。")
    except AssertionError:
        print("正常情况测试失败：断言意外触发。")


# 测试异常情况（最后一维不为4）
def test_abnormal_case():
    mock_obj = MockClass()

    num_center_objects = 3  # N
    max_num_gt = 2  # G
    num_future_timestamps = 5  # T

    # 模拟数据（最后一维为3）
    center_gt_trajs = torch.randn(num_center_objects, max_num_gt, num_future_timestamps, 3)
    center_gt_trajs_mask = torch.randint(0, 2, (num_center_objects, max_num_gt, num_future_timestamps)).bool()
    center_gt_final_valid_idx = torch.randint(0, max_num_gt, (num_center_objects,))

    mock_obj.forward_ret_dict['center_gt_trajs'] = center_gt_trajs
    mock_obj.forward_ret_dict['center_gt_trajs_mask'] = center_gt_trajs_mask
    mock_obj.forward_ret_dict['center_gt_final_valid_idx'] = center_gt_final_valid_idx

    try:
        mock_obj.get_decoder_loss()
        print("异常情况测试失败：断言未触发。")
    except AssertionError:
        print("异常情况测试通过：数据形状不符合预期，断言触发。")


if __name__ == "__main__":
    test_normal_case()
    test_abnormal_case()

tensor([[[[ 0.4539,  0.7856,  0.7430,  0.6254],
          [ 1.7027, -0.0789,  0.5867, -0.0434],
          [ 1.0199,  0.4195, -2.0023, -0.4034],
          [ 0.3729,  1.2831,  0.5861,  0.9424],
          [-0.4905, -0.3941,  0.9828, -0.7594]],

         [[-1.8375,  0.8921,  1.5435,  0.7472],
          [ 0.3211,  2.4019, -0.4251, -1.2269],
          [-1.5076,  0.0188,  0.6734, -0.2308],
          [ 0.1707,  1.3052,  0.3805,  0.1863],
          [ 0.4523,  0.0609, -1.1108, -0.7062]],

         [[ 1.4860,  0.2183, -1.4853,  1.6808],
          [-0.4053,  1.1869, -0.7784, -0.4895],
          [-0.5390,  0.3594, -0.2415, -0.6943],
          [ 0.4139,  0.0707, -0.1253,  2.1610],
          [ 0.9703, -0.6964,  0.2601,  0.5065]],

         [[ 0.7543,  0.5098, -1.3711, -0.2422],
          [-0.0788,  0.5807,  0.0357, -0.3722],
          [ 0.2144,  1.3987, -1.8032,  0.9999],
          [-0.6277, -0.3563, -0.1774,  0.0917],
          [-0.5992, -0.9964,  0.9188,  1.2907]]],


        [[[ 0.5998,  3.2261, -